# 03 - MOMFA Analysis

Deep analysis of MOMFA results: Pareto fronts, hypervolume convergence,
feature selection patterns, and best-solution predictions.

**Prerequisite:** Run MOMFA experiments first (30 seeds × 10 tickers).

In [ ]:
import sys, json, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from IPython.display import Image, display, Markdown

from utils.visualizer import (
    plot_pareto_front,
    plot_hypervolume_convergence,
    plot_feature_selection_heatmap,
    plot_predictions,
)

MOMFA_PATH = '../results/momfa/momfa.json'
TICKERS = [
    'JKH.N0000', 'COMB.N0000', 'DIAL.N0000', 'HNB.N0000',  'LOLC.N0000',
    'SAMP.N0000', 'NTB.N0000', 'HHL.N0000',  'DIST.N0000', 'HAYL.N0000',
]
FEATURE_NAMES = [
    'RSI', 'MACD', 'MACD Signal', 'Stoch %K', 'Stoch %D', 'ROC', 'Williams %R',
    'EMA9', 'EMA21', 'EMA50', 'SMA20', 'ADX', 'CCI', 'Parabolic SAR',
    'BB Upper', 'BB Lower', 'BB Width', 'ATR', 'Std Dev',
    'OBV', 'MFI', 'VWAP', 'Vol SMA', 'Vol ROC', 'CMF',
]

if not os.path.exists(MOMFA_PATH):
    print('MOMFA results not found. Run MOMFA experiments first.')
    momfa = None
else:
    with open(MOMFA_PATH) as f:
        momfa = json.load(f)
    print(f'Loaded MOMFA results for: {list(momfa.keys())}')

## 1. Pareto Fronts - All Tickers

Each point = one non-dominated solution found by MOMFA.
x-axis = model complexity (param count), y-axis = prediction error (RMSE).
Colour = number of selected features.

In [ ]:
if momfa is None:
    print('No MOMFA results.')
else:
    for ticker in TICKERS:
        if ticker not in momfa:
            continue
        # Aggregate Pareto solutions across all 30 runs
        all_solutions = []
        for run in momfa[ticker]['runs']:
            all_solutions.extend(run['pareto_front'])
        
        path = plot_pareto_front(all_solutions, ticker)
        display(Image(path))

## 2. Hypervolume Convergence

Shows how quickly the Pareto front improves over 30 iterations.
Answers OQ-002 (does alpha decay work?) and OQ-003 (is N=20, T=30 enough?).

In [ ]:
if momfa is None:
    print('No MOMFA results.')
else:
    for ticker in TICKERS:
        if ticker not in momfa:
            continue
        # hv_per_iter: shape (n_runs, n_iter)
        hv_matrix = np.array([
            run['hypervolume_per_iter']
            for run in momfa[ticker]['runs']
        ])
        path = plot_hypervolume_convergence(hv_matrix, ticker)
        display(Image(path))

## 3. Feature Selection Frequency Heatmap

Which of the 25 technical indicators are consistently selected across tickers?
High frequency = feature is important for CSE stock prediction.

In [ ]:
if momfa is None:
    print('No MOMFA results.')
else:
    # Build (25, n_tickers) selection frequency matrix
    freq_matrix = np.zeros((25, len(TICKERS)))
    
    for j, ticker in enumerate(TICKERS):
        if ticker not in momfa:
            continue
        all_masks = []
        for run in momfa[ticker]['runs']:
            for sol in run['pareto_front']:
                all_masks.append(sol['mask'])
        if all_masks:
            freq_matrix[:, j] = np.mean(all_masks, axis=0)
    
    path = plot_feature_selection_heatmap(freq_matrix)
    display(Image(path))

## 4. Best MOMFA Solutions - Summary Table

In [ ]:
if momfa is None:
    print('No MOMFA results.')
else:
    rows = []
    for ticker in TICKERS:
        if ticker not in momfa:
            continue
        # Best RMSE solution across all runs
        best = None
        for run in momfa[ticker]['runs']:
            for sol in run['pareto_front']:
                if best is None or sol['rmse'] < best['rmse']:
                    best = sol
        if best:
            hp = best.get('hyperparams', {})
            rows.append({
                'Ticker':   ticker.replace('.N0000', ''),
                'RMSE':     f"{best['rmse']:.4f}",
                'Params':   f"{best['n_params']:,}",
                'Features': sum(best['mask']),
                'Layers':   hp.get('n_layers', '-'),
                'Units':    hp.get('units', '-'),
                'LR':       hp.get('lr', '-'),
                'Lookback': hp.get('lookback', '-'),
                'Optimizer':hp.get('optimizer', '-'),
            })
    
    pd.DataFrame(rows).set_index('Ticker')

## 5. Predictions vs Actual - Best MOMFA Solution

In [ ]:
if momfa is None:
    print('No MOMFA results.')
else:
    for ticker in TICKERS:
        if ticker not in momfa:
            continue
        preds = momfa[ticker].get('best_val_preds', [])
        true  = momfa[ticker].get('best_val_true',  [])
        if preds and true:
            path = plot_predictions(np.array(true), np.array(preds), ticker=ticker)
            display(Image(path))

## 6. Answering the Open Theoretical Questions

From `docs/FINDINGS.md` - fill in answers after reviewing the results above.

In [ ]:
questions = {
    'OQ-001': 'Does Levy flight saturate sigmoid, killing feature diversity?',
    'OQ-002': 'Does alpha decay (0.5→0.20) do meaningful work in T=30?',
    'OQ-003': 'Is N=20, T=30 sufficient for 31-D multi-objective search?',
    'OQ-004': 'Is random archive guide + distance attraction coherent?',
}

print('Open Theoretical Questions - to be answered from experimental results:')
print()
for oq, q in questions.items():
    print(f'{oq}: {q}')
    print(f'  Answer: [fill in after reviewing hypervolume convergence and feature heatmap]')
    print()